In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

---

In [ ]:
# urine_raw.csv 로드 (구글 드라이브 기준)
# df = pd.read_csv('/content/drive/MyDrive/project/mini_20260109/data/processed/urine_raw.csv')

# urine_raw.csv 로드 (깃허브 기준)
df = pd.read_csv('../data/processed/urine_raw.csv')

In [ ]:
# 시간 컬럼을 날짜 형식으로 변환 (매우 중요)
df['charttime_h'] = pd.to_datetime(df['charttime_h'])

In [ ]:
# 분석 전제 확인용
print("# 행과 열 개수 확인")
print(df.shape)
print("\n# 데이터타입 확인")
print(df.dtypes)

---
## STEP 0. 분석 단위 고정 (Analysis Contract)


In [ ]:
print("[STEP 0] 분석 단위 확인")
print(f"- 전체 데이터 행 수: {len(df):,}")
print(f"- 대상 환자(Stay) 수: {df['stay_id'].nunique():,}")
print(f"- 포함된 변수: {df.columns.tolist()}")

---
## STEP 1. 데이터 시간순 정렬 확인

In [ ]:
print("[STEP 1] 시간 정렬 샘플 확인 (정상 여부)")
# 한 환자의 첫 5시간 데이터를 봅니다.
display(df.sort_values(['stay_id', 'charttime_h']).head(5))

---
## STEP 2: 시간당 측정 빈도(Data Density)

In [ ]:
print("\n[STEP 2] 시간당 데이터 밀도 확인")
stay_counts = df.groupby('stay_id').size()
print(f"- 환자 1인당 평균 관찰 시간: {stay_counts.mean():.1f}시간")

plt.figure(figsize=(10, 4))
sns.histplot(stay_counts, bins=50, kde=True, color='blue')
plt.title("Distribution of Observation Hours per Stay")
plt.xlabel("Hours")
plt.show()

## STEP 3. 주요 수치 요약 및 이상치 (Outlier)

In [ ]:
print("\n[STEP 3] 이상치 점검 (말 안 되는 수치 찾기)")
# 수치형 컬럼만 선택 (stay_id, charttime 제외)
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('stay_id')
display(df[numeric_cols].describe().T)

# 대표 변수 하나를 골라 시각화 (예: 'hr' 또는 'glucose' 등)
target_col = numeric_cols[0] # 첫 번째 수치 컬럼 자동 선택
plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x=target_col, color='salmon')
plt.title(f"Outlier Check: {target_col}")
plt.show()

In [ ]:
# 1. 이상치 기준 설정 (상위 1% 혹은 의학적 임계치 1,000ml)
outlier_threshold = 1000
outlier_df = df[df['urine_ml'] > outlier_threshold].copy()

print(f"🚩 전체 데이터 중 {outlier_threshold}ml 초과 이상치 개수: {len(outlier_df)}개")

# 2. 특정 환자군(stay_id)에 몰려 있는가?
top_outlier_patients = outlier_df['stay_id'].value_counts().head(10)

# 3. 특정 시간대(charttime_h)에 몰려 있는가?
# charttime_h가 datetime인 경우를 고려해 시간(hour) 단위로 추출
if pd.api.types.is_datetime64_any_dtype(outlier_df['charttime_h']):
    outlier_df['hour_only'] = outlier_df['charttime_h'].dt.hour
else:
    # 이미 숫자형(시간)인 경우 12시간 단위 등으로 버킷팅
    outlier_df['hour_only'] = (outlier_df['charttime_h'] // 1) % 24

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 이상치를 많이 가진 상위 환자 ID
sns.barplot(x=top_outlier_patients.index.astype(str), y=top_outlier_patients.values, ax=axes[0], palette='magma')
axes[0].set_title("Top 10 Patients with Most Urine Outliers", fontsize=12)
axes[0].set_xlabel("Stay ID")
axes[0].set_ylabel("Outlier Count")
axes[0].tick_params(axis='x', rotation=45)

# 오른쪽: 이상치가 발생하는 시간대 분포 (24시간 주기)
sns.countplot(data=outlier_df, x='hour_only', ax=axes[1], color='salmon')
axes[1].set_title("Urine Outliers by Time of Day (24h)", fontsize=12)
axes[1].set_xlabel("Hour of Day")
axes[1].set_ylabel("Outlier Count")

plt.tight_layout()
plt.show()

# 4. 요약 통계 출력
if not outlier_df.empty:
    most_freq_patient = top_outlier_patients.index[0]
    print(f"\n💡 인사이트 요약:")
    print(f"- 가장 많은 이상치를 가진 환자: ID {most_freq_patient} ({top_outlier_patients.values[0]}회 발생)")
    print(f"- 이상치 발생이 잦은 시간대: {outlier_df['hour_only'].mode()[0]}시 근처")

---
## STEP 4: 시간 흐름 패턴 확인 (Trend)

In [ ]:
# 1. 데이터 검증 및 샘플 추출
if df.empty or 'stay_id' not in df.columns:
    raise ValueError("데이터프레임이 비어있거나 stay_id 컬럼이 없습니다")

unique_stays = df['stay_id'].dropna().unique()
if len(unique_stays) == 0:
    raise ValueError("유효한 stay_id가 없습니다")

sample_id = unique_stays[0]
sample_df = df[df['stay_id'] == sample_id].sort_values('charttime_h')

# 2. 분석할 컬럼 필터링
vital_cols = ['urine_ml', 'weight_kg', 'urine_ml_kg_hr', 'oliguria_flag']
available_cols = [col for col in vital_cols if col in sample_df.columns and not sample_df[col].dropna().empty]

if len(available_cols) == 0:
    raise ValueError("시각화할 유효한 데이터가 없습니다")

# 3. 시각화 (squeeze=False로 일관된 axes 처리)
fig, axes = plt.subplots(len(available_cols), 1, figsize=(12, 2 * len(available_cols)),
                         sharex=True, squeeze=False)
axes = axes.flatten()  # 항상 1D 배열로 변환

for i, col in enumerate(available_cols):
    data = sample_df[['charttime_h', col]].dropna()

    axes[i].plot(data['charttime_h'], data[col], marker='o', markersize=4,
                linestyle='-', color='dodgerblue', linewidth=1.5)
    axes[i].set_title(f"Trend: {col}", loc='left', fontsize=10, fontweight='bold')
    axes[i].grid(True, alpha=0.3, linestyle='--')
    axes[i].set_ylabel(col, fontsize=9)

    # 마지막 subplot에만 x축 레이블 추가
    if i == len(available_cols) - 1:
        axes[i].set_xlabel("Hours since Admission (charttime_h)", fontsize=10)

plt.suptitle(f"Patient Stay ID: {sample_id}", fontsize=12, y=0.995)
plt.tight_layout()
plt.show()


---
## STEP 5 & 6. 결측치 비율 및 패턴 (Missing Map)

In [ ]:
print("\n[STEP 5/6] 결측치 비율 및 지도")
missing_pct = df[numeric_cols].isnull().mean() * 100
print(missing_pct.sort_values(ascending=False))

plt.figure(figsize=(12, 6))
# 데이터가 너무 크면 일부만 샘플링하여 시각화
sns.heatmap(df[numeric_cols].iloc[:1000].isnull(), cbar=False, cmap='viridis')
plt.title("Missing Value Map (Yellow = Missing / Purple = Data)")
plt.show()

---
## STEP 8. 결측과 상태의 관계 (Informative Missing)

In [ ]:
print("\n[STEP 8] 결측의 정보성 확인")
# 예: 특정 변수가 비어있는 시간대와 그렇지 않은 시간대의 다른 수치 비교
df['is_missing'] = df[target_col].isnull()
# 두 번째 수치 컬럼이 있다면 그것과 비교
if len(numeric_cols) > 1:
    compare_col = numeric_cols[1]
    comparison = df.groupby('is_missing')[compare_col].mean()
    print(f"'{target_col}' 결측 여부에 따른 '{compare_col}' 평균 차이:")
    print(comparison)

---
## STEP 9. 최종 판정 (Go/No-Go)

In [ ]:
print("[STEP 9] Sliding Window 투입 가능 여부 판단")
min_hours = stay_counts.min()
if min_hours >= 24: # 우리 코호트 기준이 24시간 이상 체류라면
    print(f"✅ PASS: 모든 환자가 최소 {min_hours}시간 이상의 데이터를 보유함.")
else:
    print(f"⚠️ WARNING: 데이터가 {min_hours}시간 미만인 환자가 존재함. 필터링 필요 여부 확인!")

print("\nEDA 검증 완료.")

In [ ]:
# 0. 사전 검증
required_cols = ['stay_id', 'charttime_h']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"필수 컬럼이 누락되었습니다: {missing_cols}")

# 1. 환자별 체류 시간(Duration) 계산
# charttime_h가 datetime일 경우, (최대값 - 최소값)으로 실제 체류 시간을 구합니다.
stay_summary = df.groupby('stay_id')['charttime_h'].agg(['min', 'max', 'count'])
stay_summary['max_hours'] = (stay_summary['max'] - stay_summary['min']).dt.total_seconds() / 3600
stay_summary.columns = ['first_time', 'last_time', 'record_count', 'max_hours']

short_stays = stay_summary[stay_summary['max_hours'] < 24].index

if len(short_stays) > 0:
    print(f"⚠️ {len(short_stays)}명의 환자가 24시간 미만의 데이터를 보유하고 있습니다.")
    print(f"   (전체 환자 수: {len(stay_summary)}명, 비율: {len(short_stays)/len(stay_summary)*100:.1f}%)\n")

    # 체류 시간 통계 (이제 max_hours는 숫자형이므로 정상 작동합니다)
    short_stats = stay_summary.loc[short_stays, 'max_hours']
    print(f"📊 24시간 미만 환자 체류 시간 통계:")
    print(f"   - 평균: {short_stats.mean():.2f}시간")
    print(f"   - 중앙값: {short_stats.median():.2f}시간")
    print(f"   - 최소: {short_stats.min():.2f}시간")
    print(f"   - 최대: {short_stats.max():.2f}시간\n")

    # 2. 컬럼별 유효 데이터 개수 확인 (이하 동일)
    vital_cols = ['sao2', 'ph', 'lactate', 'creatinine', 'bilirubin', 'wbc', 'platelets', 'potassium', 'sodium']
    available_cols = [col for col in vital_cols if col in df.columns]

    if available_cols:
        short_stay_df = df[df['stay_id'].isin(short_stays)]
        missing_report = short_stay_df.groupby('stay_id')[available_cols].count()

        print("📋 [컬럼별 유효 데이터 개수 - 상위 10명]")
        print(missing_report.head(10).to_string())
        print()

        print("📉 [지표별 데이터 부재 환자 통계]")
        summary_data = []
        for col in available_cols:
            no_data_count = (missing_report[col] == 0).sum()
            has_data_count = (missing_report[col] > 0).sum()
            avg_records = missing_report[col].mean()
            summary_data.append({
                '지표': col,
                '데이터 없음': no_data_count,
                '데이터 있음': has_data_count,
                '평균 레코드 수': f"{avg_records:.1f}"
            })

        summary_df = pd.DataFrame(summary_data)
        print(summary_df.to_string(index=False))
else:
    print("✅ 모든 환자가 24시간 이상의 데이터를 충실히 보유하고 있습니다.")


---
## Step 10. Report MD 생성

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ==============================================================================
# 1. Report 내용 수집
# ==============================================================================

# 기본 정보
n_rows = len(df)
n_stays = df['stay_id'].nunique()
n_cols = len(df.columns)
columns = df.columns.tolist()

# 수치형 컬럼
numeric_cols = ['urine_ml', 'weight_kg', 'urine_ml_kg_hr', 'oliguria_flag']
available_numeric = [col for col in numeric_cols if col in df.columns]

# 기술 통계
desc_stats = df[available_numeric].describe().round(2)

# 결측치
missing_data = []
for col in available_numeric:
    missing_count = df[col].isna().sum()
    missing_pct = df[col].isna().mean() * 100
    missing_data.append({
        'Column': col,
        'Missing Count': f"{missing_count:,}",
        'Percentage': f"{missing_pct:.2f}%"
    })
missing_df = pd.DataFrame(missing_data)

# 환자별 관찰 시간
stay_summary = df.groupby('stay_id')['charttime_h'].agg(['min', 'max', 'count'])
stay_summary['max_hours'] = (stay_summary['max'] - stay_summary['min']).dt.total_seconds() / 3600
avg_hours = stay_summary['max_hours'].mean()
median_hours = stay_summary['max_hours'].median()

# 24시간 미만 환자
short_stays = stay_summary[stay_summary['max_hours'] < 24]
n_short = len(short_stays)
pct_short = n_short / n_stays * 100

# Oliguria 통계
if 'oliguria_flag' in df.columns:
    oliguria_rate = df['oliguria_flag'].mean() * 100
    oliguria_count = df['oliguria_flag'].sum()
else:
    oliguria_rate = 0
    oliguria_count = 0

# ==============================================================================
# 2. Markdown Report 생성
# ==============================================================================

report = f"""# MIMIC-IV Urine Output Data EDA Report

**Analysis Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## 1. Dataset Basic Information

- **Total Records:** {n_rows:,}
- **Unique Patients (stay_id):** {n_stays:,}
- **Number of Features:** {n_cols}

### Column Names and Data Types

```
{df.dtypes.to_string()}
```

본 Urine 데이터는 {n_rows:,}개의 시간별 소변량 기록으로 구성되며, 
{n_stays:,}명의 ICU 환자에 대한 배뇨 정보를 포함한다.

Foley catheter가 삽입된 환자에서 주로 측정되며, 
시간당 소변량은 신기능 및 체액 상태의 중요한 지표이다.

## 2. Descriptive Statistics

### Numerical Variables

```
{desc_stats.to_string()}
```

주요 지표 해석:
- **urine_ml**: 시간당 소변량 (mL)
- **urine_ml_kg_hr**: 체중 보정 소변량 (mL/kg/hr)
  - 정상: 0.5-1.0 mL/kg/hr
  - 핍뇨 (Oliguria): <0.5 mL/kg/hr
- **oliguria_flag**: 핍뇨 여부 (1=핍뇨)

## 3. Missing Values Analysis

| Column | Missing Count | Percentage |
|--------|---------------|------------|
"""

for _, row in missing_df.iterrows():
    report += f"| {row['Column']} | {row['Missing Count']} | {row['Percentage']} |\n"

report += f"""
### 결측 패턴 해석

Urine 데이터의 결측은 주로 다음 원인에 기인한다:

1. **Foley catheter 미삽입**: 모든 ICU 환자가 유치도뇨관을 가지고 있지 않음
2. **측정 누락**: 간헐적 측정 또는 기록 누락

소변량 결측은 "측정 불가" 상태를 의미하므로,
Median imputation이 적절하며, 결측 자체를 별도 feature로 활용하지 않는다.

## 4. Temporal Coverage Analysis

### 환자별 Urine 데이터 시간 범위

- **평균 관찰 시간:** {avg_hours:.1f}시간
- **중앙값 관찰 시간:** {median_hours:.1f}시간

### 24시간 미만 데이터 보유 환자

- **해당 환자 수:** {n_short:,}명 ({pct_short:.1f}%)
- Foley catheter 조기 제거 또는 단기 ICU 체류로 인한 것으로 추정

## 5. Oliguria (핍뇨) Analysis

### 정의
- **Oliguria**: urine_ml_kg_hr < 0.5 mL/kg/hr

### 통계
- **전체 핍뇨 기록:** {oliguria_count:,}건 ({oliguria_rate:.2f}%)
- 핍뇨는 급성 신손상(AKI)의 조기 지표로, 중요한 예측 feature

### 임상적 의의

핍뇨는 다음 상태를 시사할 수 있다:
- 저혈량증 (Hypovolemia)
- 급성 신손상 (Acute Kidney Injury)
- 패혈성 쇼크 (Septic Shock)
- 심부전 (Heart Failure)

## 6. Clinical Reference

| 상태 | urine_ml_kg_hr | 임상적 의미 |
|------|----------------|------------|
| 정상 | >1.0 | 적절한 신관류 |
| 경계 | 0.5-1.0 | 모니터링 필요 |
| 핍뇨 | <0.5 | 신기능 저하 의심 |
| 무뇨 | <0.1 | 급성 신부전 |

## 7. Preprocessing Recommendations

### 전처리 전략

| 변수 | 권장 전략 | 근거 |
|------|----------|------|
| urine_ml | 6시간 합산 (urine_ml_6h) | 관찰 윈도우에 맞춤 |
| urine_ml_kg_hr | 6시간 평균 | 체중 보정 지표 |
| oliguria_flag | 6시간 내 1회라도 핍뇨 시 1 | 보수적 판단 |
| weight_kg | 환자별 고정값 사용 | 시간에 따른 변화 무시 |

### 슬라이딩 윈도우 적용 시 고려사항

1. **6시간 집계**: 관찰 윈도우(6h)에 맞춰 소변량 합산
2. **체중 보정**: 환자별 체중으로 정규화하여 비교 가능성 확보
3. **핍뇨 플래그**: 윈도우 내 핍뇨 발생 여부를 이진 feature로

## 8. Key Visualizations

![Urine EDA Visualizations](00_EDA_Urine_Visualizations.png)

---

## 9. Conclusion

소변량은 ICU 환자의 신기능과 체액 상태를 반영하는 중요한 생리 지표이다.

전처리 시 다음 사항을 고려해야 한다:
- 6시간 단위 집계로 관찰 윈도우에 맞춤
- 체중 보정 소변량(urine_ml_kg_hr)이 더 표준화된 지표
- 핍뇨(oliguria_flag)는 급성 신손상의 조기 경고 신호
- Foley 미삽입 환자는 결측 처리 (Median imputation)
"""

# ==============================================================================
# 3. Markdown 파일 저장
# ==============================================================================

with open('../reports/00_EDA_Urine_Report.md', 'w', encoding='utf-8') as f:
    f.write(report)
print("✓ Report saved: 00_EDA_Urine_Report.md")

In [ ]:

# ==============================================================================
# 4. Visualization 생성 및 저장
# ==============================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Urine Output Data EDA Visualizations', fontsize=14, fontweight='bold')

# 1. 환자별 관찰 시간 분포
ax1 = axes[0, 0]
sns.histplot(stay_summary['max_hours'], bins=50, kde=True, color='steelblue', ax=ax1)
ax1.set_title('Observation Hours per Patient')
ax1.set_xlabel('Hours')
ax1.set_ylabel('Count')

# 2. 시간당 소변량 분포
ax2 = axes[0, 1]
urine_data = df['urine_ml'].dropna()
# 이상치 제외 (99th percentile)
upper_limit = urine_data.quantile(0.99)
urine_filtered = urine_data[urine_data <= upper_limit]
sns.histplot(urine_filtered, bins=50, kde=True, color='teal', ax=ax2)
ax2.set_title('Hourly Urine Output Distribution')
ax2.set_xlabel('Urine (mL/hr)')

# 3. 체중 보정 소변량 분포
ax3 = axes[0, 2]
if 'urine_ml_kg_hr' in df.columns:
    urine_kg_data = df['urine_ml_kg_hr'].dropna()
    upper_limit_kg = urine_kg_data.quantile(0.99)
    urine_kg_filtered = urine_kg_data[urine_kg_data <= upper_limit_kg]
    sns.histplot(urine_kg_filtered, bins=50, kde=True, color='coral', ax=ax3)
    ax3.axvline(x=0.5, color='red', linestyle='--', label='Oliguria (<0.5)')
    ax3.set_title('Weight-Adjusted Urine Output')
    ax3.set_xlabel('Urine (mL/kg/hr)')
    ax3.legend()

# 4. Oliguria 비율 파이차트
ax4 = axes[1, 0]
if 'oliguria_flag' in df.columns:
    oliguria_counts = df['oliguria_flag'].value_counts()
    labels = ['Normal', 'Oliguria']
    colors = ['#27ae60', '#e74c3c']
    ax4.pie(oliguria_counts, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax4.set_title('Oliguria Distribution')

# 5. 결측률 바 차트
ax5 = axes[1, 1]
missing_pcts = df[available_numeric].isna().mean() * 100
colors = ['#e74c3c' if x > 50 else '#f39c12' if x > 20 else '#27ae60' for x in missing_pcts]
bars = ax5.barh(missing_pcts.index, missing_pcts.values, color=colors)
ax5.set_title('Missing Value Percentage')
ax5.set_xlabel('Missing %')

# 6. 체중 분포
ax6 = axes[1, 2]
if 'weight_kg' in df.columns:
    weight_data = df.groupby('stay_id')['weight_kg'].first().dropna()
    sns.histplot(weight_data, bins=50, kde=True, color='purple', ax=ax6)
    ax6.set_title('Patient Weight Distribution')
    ax6.set_xlabel('Weight (kg)')

plt.tight_layout()
plt.savefig('../reports/00_EDA_Urine_Visualizations.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Visualization saved: 00_EDA_Urine_Visualizations.png")

print("\n" + "="*50)
print("Urine EDA Report 생성 완료!")
print("="*50)